<a href="https://colab.research.google.com/github/farahnda/Learning/blob/main/Ensemble_w_Keras_%26_SKlearn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [62]:
import sklearn
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

In [63]:
X, y = make_moons(n_samples=500, noise=0.30, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

In [64]:
log_clf = LogisticRegression(solver="lbfgs", random_state=42)
rnd_clf = RandomForestClassifier(n_estimators=150, random_state=42)
svm_clf = SVC(gamma="scale", random_state=42, probability=True)

from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense

In [65]:
def build_nn():
  model = Sequential([Dense(50, activation='relu', input_shape=[2]), Dense(1, activation='sigmoid')])
  model.compile(optimizer='Adam', loss='binary_crossentropy', metrics=['accuracy'])
  return model

In [66]:
import tensorflow as tf
!pip install scikeras
from scikeras.wrappers import KerasClassifier
keras_clf = KerasClassifier(model=build_nn, epochs=500, verbose=False)
# keras_clf = tf.keras.wrappers.scikit_learn.KerasClassifier(build_nn, epochs=500, verbose=False)

In [67]:
keras_clf.estimator_type = 'classifier'

In [68]:
voting = VotingClassifier(estimators=[('lr', log_clf), ('rf', rnd_clf), ('svc', svm_clf)], voting='soft',
                          # ('keras', keras_clf)
                          flatten_transform=True)

In [71]:
for clf in (log_clf, rnd_clf, svm_clf, voting):
  clf.fit(X_train, y_train)
  y_pred = clf.predict(X_test)
  print(clf.__class__.__name__, accuracy_score(y_test, y_pred))

LogisticRegression 0.864
RandomForestClassifier 0.904
SVC 0.896
VotingClassifier 0.92
